# Module 5.5 — Profiling & Graph Optimisation

**Goal:** Profile the DeskMate encoder ONNX model to find the single slowest operator, apply a targeted fix (ORT transformer optimizer + thread tuning), and measure the before/after latency improvement. No guessing — measure first, fix second.

## 1. Install Dependencies

In [ ]:
# !pip install -q onnx onnxruntime onnxconverter-common onnxsim pandas matplotlib
# !pip install -q onnxruntime-tools  # provides onnxruntime.transformers.optimizer

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SMOKE_TEST = True  # set False to run real profiling with a loaded ONNX model
ENCODER_ONNX = 'models/deskmate_encoder.onnx'
ENCODER_OPT  = 'models/deskmate_encoder_opt.onnx'
ENCODER_FP16 = 'models/deskmate_encoder_fp16.onnx'
N_BENCH = 50
SEQ_LEN = 128

os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Baseline Session — Median Latency

In [ ]:
import onnxruntime as ort

def make_dummy_inputs(batch=1, seq=SEQ_LEN):
    return {
        'input_ids':      np.ones((batch, seq), dtype=np.int64),
        'attention_mask': np.ones((batch, seq), dtype=np.int64),
    }

def run_bench(session, inputs, n=N_BENCH):
    # Warmup
    for _ in range(5):
        session.run(None, inputs)
    # Measure
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        session.run(None, inputs)
        times.append((time.perf_counter() - t0) * 1000)  # ms
    return round(float(np.median(times)), 2)

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating baseline session.')
    print('[Simulated] Baseline median latency: 45.2 ms')
    baseline_ms = 45.2
    sess_base = None
    dummy_inputs = make_dummy_inputs()
else:
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.intra_op_num_threads = 4
    sess_base = ort.InferenceSession(
        ENCODER_ONNX, sess_opts, providers=['CPUExecutionProvider'])
    dummy_inputs = make_dummy_inputs()
    baseline_ms = run_bench(sess_base, dummy_inputs)
    print(f'Baseline median latency: {baseline_ms} ms')

## 3. Profiling Session — Chrome Trace JSON

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — using synthetic profile data.')
    profile_path = None
    # Synthetic profile events matching a typical DeBERTa profile
    synthetic_profile = [
        {'cat': 'Node', 'dur': 12400, 'args': {'op_name': 'MatMul'}},
        {'cat': 'Node', 'dur': 2100,  'args': {'op_name': 'Softmax'}},
        {'cat': 'Node', 'dur': 1800,  'args': {'op_name': 'Add'}},
        {'cat': 'Node', 'dur': 1600,  'args': {'op_name': 'LayerNormalization'}},
        {'cat': 'Node', 'dur':  900,  'args': {'op_name': 'Gather'}},
        {'cat': 'Node', 'dur':  620,  'args': {'op_name': 'Mul'}},
        {'cat': 'Node', 'dur':  400,  'args': {'op_name': 'Gelu'}},
        {'cat': 'Node', 'dur':  300,  'args': {'op_name': 'Reshape'}},
        {'cat': 'Node', 'dur':  180,  'args': {'op_name': 'Transpose'}},
        {'cat': 'Node', 'dur':   90,  'args': {'op_name': 'Cast'}},
        # Duplicate entries simulate multiple layer occurrences
        {'cat': 'Node', 'dur': 11200, 'args': {'op_name': 'MatMul'}},
        {'cat': 'Node', 'dur': 1950,  'args': {'op_name': 'Softmax'}},
        {'cat': 'Node', 'dur': 1700,  'args': {'op_name': 'Add'}},
        {'cat': 'Node', 'dur': 1550,  'args': {'op_name': 'LayerNormalization'}},
        {'cat': 'Other', 'dur': 500,  'args': {'op_name': 'MemcpyH2D'}},
    ]
    events = synthetic_profile
else:
    prof_opts = ort.SessionOptions()
    prof_opts.enable_profiling = True
    prof_opts.profile_file_prefix = 'ort_profile'
    prof_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    prof_opts.intra_op_num_threads = 4
    sess_prof = ort.InferenceSession(
        ENCODER_ONNX, prof_opts, providers=['CPUExecutionProvider'])
    # Run one measured pass after warmup
    sess_prof.run(None, dummy_inputs)
    profile_path = sess_prof.end_profiling()
    print('Profile written to:', profile_path)
    with open(profile_path) as f:
        events = json.load(f)

## 4. Parse Profile — Rank Ops by Duration

In [ ]:
nodes = [
    {'op': e['args'].get('op_name', e.get('name', 'unknown')), 'dur_us': e['dur']}
    for e in events
    if e.get('cat') == 'Node' and 'dur' in e
]

df_prof = pd.DataFrame(nodes)
df_ranked = (df_prof.groupby('op')['dur_us']
             .sum()
             .sort_values(ascending=False)
             .reset_index())
total_us = df_ranked['dur_us'].sum()
df_ranked['pct'] = (df_ranked['dur_us'] / total_us * 100).round(1)
df_ranked['dur_ms'] = (df_ranked['dur_us'] / 1000).round(2)

print('Top-10 ops by total execution time:')
print(df_ranked[['op', 'dur_ms', 'pct']].head(10).to_string(index=False))

slowest_op  = df_ranked.iloc[0]['op']
slowest_pct = df_ranked.iloc[0]['pct']
slowest_ms  = df_ranked.iloc[0]['dur_ms']
print(f'\nSlowest op: {slowest_op} — {slowest_pct}% of total ({slowest_ms} ms)')

## 5. Flame Chart — Perfetto / Chrome Tracing

ORT's profile output is a Chrome trace JSON. To view it as a flame chart:

1. Open [https://ui.perfetto.dev](https://ui.perfetto.dev) in your browser
2. Click **Open trace file** and select the `ort_profile_*.json` file
3. Use W/S to zoom in/out, A/D to pan
4. Click any bar to see op name, duration, and arguments

Alternatively, open `chrome://tracing` in Chrome and drag the file in.

**What to look for:**
- The widest bar = your bottleneck
- Gaps between bars = Python/ORT framework overhead
- Many narrow bars = fragmentation — look for fusion opportunities

## 6. Top-10 Ops Bar Chart

In [ ]:
top10 = df_ranked.head(10)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top10['op'][::-1], top10['dur_ms'][::-1], color='steelblue')
ax.set_xlabel('Total time (ms)')
ax.set_title('Encoder Profile: Time per Operator Type')
for bar, pct in zip(bars, top10['pct'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('profile_ops.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: profile_ops.png')

## 7. Fix 1 — ORT Transformer Optimizer

The `onnxruntime.transformers.optimizer` knows BERT/DeBERTa architecture patterns and applies: Attention fusion, LayerNorm fusion, GELU fusion, and Embedding fusion. This typically reduces op count by 30–50% and latency by 15–30% on CPU.

In [ ]:
if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating ORT transformer optimization.')
    print('[Simulated] Optimised ONNX saved to', ENCODER_OPT)
    print('[Simulated] Nodes before: 312   Nodes after: 198')
    opt_ms = 31.4
    print(f'[Simulated] Optimised latency: {opt_ms} ms')
else:
    from onnxruntime.transformers import optimizer

    opt_model = optimizer.optimize_model(
        ENCODER_ONNX,
        model_type='bert',       # DeBERTa uses BERT-like patterns
        num_heads=12,
        hidden_size=768,
    )
    opt_model.save_model_to_file(ENCODER_OPT)
    print('Optimised model saved to:', ENCODER_OPT)

    # Count nodes
    import onnx
    n_before = len(onnx.load(ENCODER_ONNX).graph.node)
    n_after  = len(onnx.load(ENCODER_OPT).graph.node)
    print(f'Nodes before: {n_before}   Nodes after: {n_after}')

    sess_opt = ort.InferenceSession(
        ENCODER_OPT,
        ort.SessionOptions(),
        providers=['CPUExecutionProvider'])
    opt_ms = run_bench(sess_opt, dummy_inputs)
    print(f'Optimised latency: {opt_ms} ms')

print(f'Speedup from fusion: {round(baseline_ms / opt_ms, 2)}x  '
      f'({round((1 - opt_ms/baseline_ms)*100, 1)}% reduction)')

## 8. Fix 2 — Thread Count Tuning

In [ ]:
thread_counts = [1, 2, 4, 8]

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating thread tuning.')
    thread_results = {1: 52.1, 2: 38.4, 4: 31.4, 8: 33.9}
    print('Threads | Latency (ms)')
    for t, ms in thread_results.items():
        marker = ' <-- best' if ms == min(thread_results.values()) else ''
        print(f'  {t:2d}    |   {ms}{marker}')
else:
    thread_results = {}
    for n_threads in thread_counts:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = n_threads
        sess_t = ort.InferenceSession(
            ENCODER_OPT, opts, providers=['CPUExecutionProvider'])
        lat = run_bench(sess_t, dummy_inputs)
        thread_results[n_threads] = lat
        print(f'  {n_threads} threads: {lat} ms')

best_threads = min(thread_results, key=thread_results.get)
best_thread_ms = thread_results[best_threads]
print(f'\nBest: {best_threads} threads at {best_thread_ms} ms')

## 9. Thread Tuning Chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(thread_results.keys()), list(thread_results.values()),
        marker='o', color='steelblue')
ax.axvline(best_threads, color='coral', linestyle='--', label=f'Best: {best_threads} threads')
ax.set_xlabel('intra_op_num_threads')
ax.set_ylabel('Median latency (ms)')
ax.set_title('Encoder Latency vs Thread Count')
ax.legend()
plt.tight_layout()
plt.savefig('profile_threads.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: profile_threads.png')

## 10. Fix 3 — fp16 Graph Conversion (GPU)

In [ ]:
# fp16 MatMul runs ~2x faster than fp32 on NVIDIA Tensor Cores.
# keep_io_types=True: inputs/outputs stay fp32 (callers unchanged),
# internal weights and activations are cast to fp16.

if SMOKE_TEST:
    print('SMOKE_TEST=True — simulating fp16 conversion.')
    print('[Simulated] fp16 model saved to', ENCODER_FP16)
    fp16_ms = 18.2
    print(f'[Simulated] fp16 GPU latency: {fp16_ms} ms')
else:
    import torch
    if not torch.cuda.is_available():
        print('No GPU — skipping fp16 GPU benchmark.')
        fp16_ms = None
    else:
        import onnx
        from onnxconverter_common import float16

        model_proto = onnx.load(ENCODER_OPT)
        model_fp16  = float16.convert_float_to_float16(model_proto, keep_io_types=True)
        onnx.save(model_fp16, ENCODER_FP16)
        print('fp16 model saved to:', ENCODER_FP16)

        sess_fp16 = ort.InferenceSession(
            ENCODER_FP16,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        fp16_ms = run_bench(sess_fp16, dummy_inputs)
        print(f'fp16 GPU latency: {fp16_ms} ms')

## 11. Before / After Latency Comparison

In [ ]:
labels = ['Baseline\n(fp32 CPU)', 'ORT Optimizer\n(fp32 CPU)',
          f'ORT Opt + {best_threads}T\n(fp32 CPU)', 'fp16\n(GPU)']
values = [baseline_ms, opt_ms, best_thread_ms,
          fp16_ms if fp16_ms is not None else 0]
colors = ['steelblue', 'coral', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, values, color=colors)
ax.set_ylabel('Median latency (ms) — lower is better')
ax.set_title('Encoder Latency: Before vs After Optimisation')
for bar, val in zip(bars, values):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val} ms', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('profile_before_after.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: profile_before_after.png')

## 12. Checkpoint: Slowest Op + Fix Applied

In [ ]:
speedup_fusion  = round(baseline_ms / opt_ms, 2)
speedup_threads = round(baseline_ms / best_thread_ms, 2)
speedup_total   = round(baseline_ms / best_thread_ms, 2)

checkpoint = f'''
Checkpoint answer:

1. Slowest op: {slowest_op} — {slowest_pct}% of total CPU time ({slowest_ms} ms)

2. Diagnosis: {slowest_op} is the expected dominant cost in a transformer encoder.
   At fp32 with default graph, BERT-family attention and FFN MatMuls are
   not fused and run with generic PyTorch-exported kernels.

3. Fix applied:
   a) onnxruntime.transformers.optimizer — fused Attention, LayerNorm, GELU;
      node count reduced; baseline {baseline_ms} ms -> {opt_ms} ms ({speedup_fusion}x)
   b) intra_op_num_threads tuning — best at {best_threads} threads:
      {opt_ms} ms -> {best_thread_ms} ms ({speedup_threads}x vs baseline)

4. Result: {round((1 - best_thread_ms/baseline_ms)*100, 1)}% latency reduction
   from {baseline_ms} ms to {best_thread_ms} ms
'''
print(checkpoint)

## 13. Save Report

In [ ]:
report = [
    '# Profiling & Graph Optimisation Report\n',
    f'Model: {ENCODER_ONNX}',
    f'Smoke test: {SMOKE_TEST}',
    f'Benchmark runs: {N_BENCH}\n',
    '## Slowest Operators (before optimisation)',
    '',
    df_ranked[['op','dur_ms','pct']].head(5).to_markdown(index=False),
    '',
    '## Latency: Before vs After',
    '',
    '| Configuration | Latency (ms) | Speedup vs baseline |',
    '|--------------|-------------|---------------------|',
    f'| Baseline (fp32 CPU, 4T) | {baseline_ms} | 1.00x |',
    f'| ORT Optimizer (fp32 CPU) | {opt_ms} | {speedup_fusion}x |',
    f'| ORT Opt + {best_threads} threads | {best_thread_ms} | {speedup_threads}x |',
    f'| fp16 GPU | {fp16_ms} | {round(baseline_ms/fp16_ms, 2) if fp16_ms else "N/A"}x |',
    '',
    '## Checkpoint',
    '',
    f'Single slowest op: **{slowest_op}** ({slowest_pct}% of total).',
    f'Fix: ORT transformer optimizer (fusion) + {best_threads}-thread tuning.',
    f'Result: {round((1-best_thread_ms/baseline_ms)*100,1)}% latency reduction.',
]

with open('reports/profile_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/profile_report.md')
print('\n=== Module 5.5 Complete ===')
print(f'Latency: {baseline_ms} ms -> {best_thread_ms} ms  '
      f'({round((1-best_thread_ms/baseline_ms)*100,1)}% improvement)')